# SHAP Interpretability

TreeSHAP explanations for the Optuna-tuned LightGBM model on engineered features.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from house_price_predictor import (
    build_feature_frame, load_housing_data, tune_lightgbm_optuna, LightGBMRegressor, HAS_OPTUNA
)
from house_price_predictor.shap_explain import explain_tree_model, save_shap_summary, save_shap_bar

train, _ = load_housing_data()
eng, _ = build_feature_frame(train, oof_target_encoding=True, include_onehot=True)
X, y = eng.X, eng.y
if HAS_OPTUNA:
    model, params, cv = tune_lightgbm_optuna(X, y, n_trials=12, n_splits=3, timeout=60, n_estimators=600)
    print("CV RMSE", cv, params)
else:
    model = LightGBMRegressor(n_estimators=400)
    model.fit(X, y)
explainer, shap_values, X_s, names = explain_tree_model(model, X, eng.schema.feature_names)
save_shap_summary(shap_values, X_s, eng.schema.feature_names, ROOT / "artifacts" / "shap_summary.png")
save_shap_bar(shap_values, eng.schema.feature_names, ROOT / "artifacts" / "shap_bar.png")
print("Wrote artifacts/shap_summary.png and shap_bar.png")
